In [10]:
# IMPORTS

from bs4 import BeautifulSoup
import bs4
import requests
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

import pandas as pd
import numpy as np
from pathlib import Path

In [11]:
# USING SELENIUM FOR WEB SCRAPING

chrome_options = Options()
chrome_options.add_argument("--headless=new")

driver = webdriver.Chrome(options=chrome_options)

In [12]:
# LIST OF UNIQUE CATEGORIES THAT WAS NEEDED FOR THE DATASET.
# ADD OR SUBTRACT LABELS AS NEEDED

job_list = ['HR', 'designer', 'Information-Technology',
       'Teacher', 'Advocate', 'Business-Development',
       'Healthcare', 'Fitness', 'Agriculture', 'BPO', 'Sales', 'Consultant',
       'Digital-Media', 'Automobile', 'Chef', 'Finance',
       'Apparel', 'Engineering', 'Accountant', 'Construction',
       'Public-Relations', 'Banking', 'Arts', 'Aviation']

In [13]:
# CREATING A DATAFRAME TO STORE THE LINKS OF EACH INDIVIDUAL RESUME EXAMPLE

resume_links = pd.DataFrame()
category = []
link = []

In [14]:
# LOOP TO SEARCH FOR RESUME EXAMPLES FROM THE WEBPAGE AS PER THE LABELS DEFINED ABOVE. ONLY (10*12) 120 RESUMES WILL BE COLLECTED.

# REMOVE  "&bg=85&eg=100&comp=&mod="  TO EXPAND THE SEARCH 
# (bg=85 & eg=100) FILTERS THE RESUME BASED ON SCORE OF BETWEEEN 85 AND 100, INCREASE RANGE TO EXPAND SEARCH

for job in job_list:
    JOB = job.lower()
    for i in range(1,13):   # INCREASE THE RANGE TO GET MORE RESUME DATA
        PAGE = str(i)
        URL = "https://www.livecareer.com/resume-search/search?jt=" + JOB + "&bg=85&eg=100&comp=&mod=&pg=" + PAGE
        driver.get(URL)
        aTagsInLi = driver.find_elements(By.CSS_SELECTOR, 'li a')
        for a in aTagsInLi:
            if a.get_attribute('rel') == "ugc":
                category.append(JOB)
                link.append(a.get_attribute('href'))
    

In [15]:
# STORES THE CATEGORY AND LINK TO THE RESUME WEBPAGE

resume_links["Category"] = category
resume_links["link"] = link

In [16]:
# HASHES THE LINK AND CREATES AN UNIQUE ID FOR THE ROWS

import hashlib
def id(x):
    return int(hashlib.md5(x.encode('utf-8')).hexdigest(), 16)

resume_links["id"] = resume_links["link"].apply(id)

In [17]:
# NUMBER OF RESUMES COLLECTED PER DEFINED LABELS

resume_links.Category.value_counts()

Series([], Name: count, dtype: int64)

In [18]:
df = resume_links.copy()
df["Resume"] = ""
df["Raw_html"] = ""

In [20]:
# LOOP TO COLLECT THE RESUME DATA PRESENT IN THE LINKS AS COLECTED ABOVE
# STORED IN HTML AS WELL AS STRING FORMAT

for i in range(df.shape[0]):
    url = df.link[i]
    driver.get(url)
#     time.sleep(0.5)                  #ADDED DELAY, CAN BE REMOVED
    x = driver.page_source
    x = x.replace(">","> ")
    soup = bs4.BeautifulSoup(x, 'html.parser')
    div = soup.find("div", {"id": "document"})
    df.Raw_html[i] = div
    try:
        df.Resume[i] = div.text
    except:
#         ADD EXCEPTION IF REQUIRED
        pass



In [21]:
df.head()

,Category,link,id,Resume,Raw_html


In [22]:
output_path = Path("data/raw/resumes.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)